In [5]:
import re
import hashlib
import time
import requests
import pandas as pd
from pathlib import Path
from io import BytesIO
from urllib.parse import unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from tqdm import tqdm

DATA_DIR = Path("../data")
df = pd.read_csv(DATA_DIR / "raw" / "chanel_raw.csv")
print(df.shape)

(1428, 17)


In [ ]:
#fonctions de téléchargement des images
HEADERS = {"User-Agent": "Mozilla/5.0"}

def url_to_name(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest() + ".jpg"

def extract_original_url(url: str):
    """Pour une URL Cloudinary .../image/fetch/<transfos>/<url_origine>, renvoie l'url d'origine."""
    idx = url.find("/fetch/")
    if idx == -1:
        return None
    rest = url[idx + len("/fetch/"):]
    m = re.search(r"https?(?::|%3A)(?://|%2F%2F)", rest, flags=re.I)
    if not m:
        return None
    return unquote(rest[m.start():])

def try_get(url, retries=2, timeout=15):
    """Renvoie (bytes|None, http_code|None, message_erreur|None)."""
    last_code, last_err = None, None
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.content, 200, None
            last_code, last_err = r.status_code, f"HTTP {r.status_code}"
            if r.status_code in (400, 403, 404, 410):   # inutile de réessayer
                break
        except Exception as e:
            last_code, last_err = None, str(e)[:200]
        time.sleep(1)
    return None, last_code, last_err

def to_rgb_white(im):
    """Convertit en RGB en remplaçant la transparence par du blanc."""
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA")
        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
        bg.alpha_composite(im)
        return bg.convert("RGB")
    return im.convert("RGB")

def download(url: str):
    path = IMG_DIR / url_to_name(url)
    if path.exists():
        return {"imageurl": url, "status": "ok", "source": "cache", "http_code": 200, "error": None}

    sources = [("direct", url)]
    orig = extract_original_url(url)
    if orig:
        sources.append(("fallback", orig))

    code, err = None, None
    for source, u in sources:
        content, code, err = try_get(u)
        if content is None:
            continue
        try:
            img = to_rgb_white(Image.open(BytesIO(content)))
            img.save(path, "JPEG", quality=95)
            return {"imageurl": url, "status": "ok", "source": source, "http_code": 200, "error": None}
        except Exception as e:
            code, err = None, f"image invalide: {str(e)[:100]}"
    return {"imageurl": url, "status": "error", "source": None, "http_code": code, "error": err}

In [ ]:
#téléchargement complet de toutes les images
results = []
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = [ex.submit(download, u) for u in urls]
    for f in tqdm(as_completed(futures), total=len(futures)):
        results.append(f.result())

log = pd.DataFrame(results)
log.to_csv(DATA_DIR / "raw" / "download_log.csv", index=False)

print(log["status"].value_counts())
print("\nSource des succès :")
print(log[log.status == "ok"]["source"].value_counts())
print("\nCodes d'erreur :")
print(log[log.status == "error"]["http_code"].value_counts(dropna=False))

  5%|▍         | 68/1383 [00:06<02:01, 10.84it/s]c:\Users\ghile_v5oqjjq\Desktop\Master 2\Indexation de données massives\Projet_MAZARI_SELMANE\venv\Lib\site-packages\PIL\Image.py:1134: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 1383/1383 [03:23<00:00,  6.80it/s]

status
ok       918
error    465
Name: count, dtype: int64

Source des succès :
source
direct      889
fallback     15
cache        14
Name: count, dtype: int64

Codes d'erreur :
http_code
404    464
400      1
Name: count, dtype: int64


In [12]:
#on lie charque ligne à son image
def is_valid_image(p):
    try:
        with Image.open(p) as im:
            im.verify()
        return True
    except Exception:
        return False

df["image_path"] = df["imageurl"].apply(
    lambda u: str(IMG_DIR / url_to_name(u)) if pd.notna(u) else None
)
df["image_ok"] = df["image_path"].apply(
    lambda p: p is not None and Path(p).exists() and is_valid_image(p)
)

print(df["image_ok"].value_counts())
print(f"Taux d'images disponibles : {df['image_ok'].mean():.2%}")

df.to_csv(DATA_DIR / "raw" / "chanel_with_paths.csv", index=False)

image_ok
True     918
False    510
Name: count, dtype: int64
Taux d'images disponibles : 64.29%


In [14]:
#images manquantes par categorie
pd.set_option("display.max_rows", None)

missing = (
    df.groupby("category2_code")["image_ok"]
      .agg(nb_images_base="size", nb_images_dispo="sum", taux="mean")
)
missing["nb_manquantes"] = missing["nb_images_base"] - missing["nb_images_dispo"]
missing = missing.sort_values("taux")

display(missing)

,nb_images_base,nb_images_dispo,taux,nb_manquantes
category2_code,,,,
1.5 1 CAMELIA. 5 ALLURES,3,0,0.000000,3
1932 COLLECTION (2022),3,0,0.000000,3
BODY,1,0,0.000000,1
COMETE,5,0,0.000000,5
LES EXCLUSIFS DE CHANEL,3,0,0.000000,3
LES INTEMPORELS DE CHANEL,4,0,0.000000,4
MASSAGE ACCESSORIES,1,0,0.000000,1
L'ESPRIT DU LION,1,0,0.000000,1
SUN PROTECTION,1,0,0.000000,1
